# Principio de Inversión de Dependencias (DIP) — Sistema de citas médicas

## Introducción
El Principio de Inversión de Dependencias (DIP) establece que los módulos de alto nivel no deben depender de módulos de bajo nivel concretos, sino de abstracciones. La implementación concreta se **inyecta** desde afuera, en vez de ser instanciada directamente dentro de la clase de alto nivel.

## Objetivos
- Mostrar una clase de alto nivel (`GestorCitas`) que instancia directamente una implementación concreta de notificación.
- Rediseñarla para que dependa de una abstracción y reciba la implementación inyectada, pudiendo sustituirla sin modificar `GestorCitas`.

## Ejemplo que viola el DIP

`GestorCitas` (módulo de alto nivel: coordina el agendamiento) crea directamente un `NotificadorSMS` (módulo de bajo nivel concreto) dentro de su propio constructor.

In [1]:
class NotificadorSMS:
    def __init__(self, proveedor: str = "Twilio") -> None:
        self.proveedor = proveedor
        self.enviados = 0

    def enviar_sms(self, destinatario: str, mensaje: str) -> str:
        self.enviados += 1
        texto = f"[SMS via {self.proveedor}] Para {destinatario}: {mensaje}"
        print(texto)
        return texto


class GestorCitas:
    def __init__(self) -> None:
        self.notificador = NotificadorSMS()  # Dependencia concreta creada dentro de la clase de alto nivel
        self.citas_agendadas = []

    def agendar_cita(self, paciente: str, fecha: str) -> None:
        self.citas_agendadas.append((paciente, fecha))
        self.notificador.enviar_sms(paciente, f"Tu cita quedó agendada para el {fecha}")

    def total_citas(self) -> int:
        return len(self.citas_agendadas)

In [2]:
gestor = GestorCitas()
gestor.agendar_cita("Laura Gómez", "2026-09-02")
print("Total de citas:", gestor.total_citas())

[SMS via Twilio] Para Laura Gómez: Tu cita quedó agendada para el 2026-09-02
Total de citas: 1


### Por qué esto es un problema

Si la clínica quiere notificar por correo electrónico en vez de SMS (o además de SMS), la única forma es **editar el código fuente de `GestorCitas`** para instanciar otra clase. `GestorCitas`, que debería ocuparse solo de coordinar el agendamiento, terminó acoplado a los detalles concretos de cómo se envía un SMS.

## Versión corregida: dependiendo de una abstracción

Se define la abstracción `Notificador`. `GestorCitas` recibe cualquier `Notificador` por constructor (inyección de dependencias) y nunca instancia una implementación concreta.

In [3]:
from abc import ABC, abstractmethod


class Notificador(ABC):
    @abstractmethod
    def enviar(self, destinatario: str, mensaje: str) -> str: ...


class NotificadorSMS(Notificador):
    def __init__(self, proveedor: str = "Twilio") -> None:
        self.proveedor = proveedor
        self.enviados = 0

    def enviar(self, destinatario: str, mensaje: str) -> str:
        self.enviados += 1
        texto = f"[SMS via {self.proveedor}] Para {destinatario}: {mensaje}"
        print(texto)
        return texto


class NotificadorEmail(Notificador):
    def __init__(self, servidor_smtp: str = "smtp.clinica.com") -> None:
        self.servidor_smtp = servidor_smtp
        self.enviados = 0

    def enviar(self, destinatario: str, mensaje: str) -> str:
        self.enviados += 1
        texto = f"[Email via {self.servidor_smtp}] Para {destinatario}: {mensaje}"
        print(texto)
        return texto


class GestorCitas:
    def __init__(self, notificador: Notificador) -> None:
        self.notificador = notificador  # Recibido por inyección: GestorCitas solo conoce la abstracción
        self.citas_agendadas = []

    def agendar_cita(self, paciente: str, fecha: str) -> None:
        self.citas_agendadas.append((paciente, fecha))
        self.notificador.enviar(paciente, f"Tu cita quedó agendada para el {fecha}")

    def total_citas(self) -> int:
        return len(self.citas_agendadas)

### Demostración: sustituir la implementación sin tocar `GestorCitas`

In [4]:
gestor_sms = GestorCitas(notificador=NotificadorSMS())
gestor_sms.agendar_cita("Laura Gómez", "2026-09-02")

gestor_email = GestorCitas(notificador=NotificadorEmail())
gestor_email.agendar_cita("Pedro Martínez", "2026-09-03")

print("Total citas (SMS):", gestor_sms.total_citas())
print("Total citas (Email):", gestor_email.total_citas())

assert gestor_sms.total_citas() == 1
assert gestor_email.total_citas() == 1
assert isinstance(gestor_sms.notificador, Notificador)
assert isinstance(gestor_email.notificador, Notificador)
print("¡Todo correcto! GestorCitas no cambió entre un caso y otro.")

[SMS via Twilio] Para Laura Gómez: Tu cita quedó agendada para el 2026-09-02
[Email via smtp.clinica.com] Para Pedro Martínez: Tu cita quedó agendada para el 2026-09-03
Total citas (SMS): 1
Total citas (Email): 1
¡Todo correcto! GestorCitas no cambió entre un caso y otro.


### Análisis

- `GestorCitas` (alto nivel) depende únicamente de `Notificador` (abstracción), nunca de `NotificadorSMS` ni `NotificadorEmail` directamente.
- Las implementaciones concretas (`NotificadorSMS`, `NotificadorEmail`) también dependen de la abstracción, no al revés: se invierte la dirección de la dependencia respecto al ejemplo original.
- Cambiar el canal de notificación es tan simple como inyectar otra implementación en el constructor; `GestorCitas` no se modifica en absoluto.

## Autoevaluación
- ¿Qué pasaría si quisiera notificar por SMS y por email al mismo tiempo? ¿Cómo encajaría esa necesidad en este diseño sin romper el DIP?

## Referencias
- Martin, R. C. — *The Dependency Inversion Principle*.
- [SOLID Principles en Python – Real Python](https://realpython.com/solid-principles-python/)